# CURC VIIRS SNPP Workflow Planning

This notebook illustrates the CURC-specific workflow layer as a thin interactive front end over `workflows/curc`.

The intended user model is:
- choose a sensor, platform, tile, and water year,
- inspect discovered inputs and planned workflow steps,
- run one step at a time or eventually submit distributed Slurm jobs.

For this example, the source reflectance tree is the real SNPP `VNP09GA` archive on `/pl/active/rittger_ops/vnp09ga.002`.

In [1]:
from __future__ import annotations

from dataclasses import asdict
import json
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "spires").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f"repo_root={REPO_ROOT}")

from spires.sensors.registry import list_supported_sensor_platforms
from workflows.curc.config import CurcWorkflowConfig, SlurmProfile
from workflows.curc.discovery import discover_viirs_snpp_water_year_reflectance_files
from workflows.curc.paths import ancillary_dir, log_root, output_raw_water_year_root, reflectance_dir, r0_dir
from workflows.curc.runner import plan_viirs_snpp_steps


repo_root=/projects/ropa5718/SpiPy


## Configuration

The notebook should not define separate workflow logic, but it should expose the user-editable parameters clearly. This cell mirrors what a future external config file will contain.

In [2]:
supported = list_supported_sensor_platforms()
print(json.dumps(supported, indent=2))
print()

# User-editable workflow parameters.
SCRATCH_ROOT = Path("/scratch/alpine/ropa5718/spipy")
INPUT_SOURCE_ROOT = Path("/pl/active/rittger_ops/vnp09ga.002")
SENSOR = "viirs"
PLATFORMS = ("snpp",)
TILES = ("h08v05",)
WATER_YEARS = (2023,)
DATES = ()
DATE_GLOB = "*"
DRY_RUN = True
APPLY_VALID_INVERSION_MASK = False
SLURM_EXCLUDE_NODES = (
    "bgpu-biokem1,bgpu-biokem2,bgpu-biokem3,bgpu-bortz1,bgpu-curc2,bgpu-curc4,"
    "bgpu-g4-18,bgpu-g4-u20,bgpu-g4-u24,bgpu-g6-u20,bgpu-ivc,bhpc-c5-u31-1,"
    "blanca-g4-u14-3,bmem-rico1,bnode0108,bgpu-g6-u34,bhpc-c5-u7-12,bnode0414,"
    "bnode0508,bhpc-c7-u7-8,bgpu-g6-u25,bgpu-chbe-rdi1"
)
SLURM_EXCLUDE_ARG = f"--exclude={SLURM_EXCLUDE_NODES}"
SLURM_MEM = "8G"

config = CurcWorkflowConfig(
    scratch_root=SCRATCH_ROOT,
    input_source_root=INPUT_SOURCE_ROOT,
    sensor=SENSOR,
    platforms=PLATFORMS,
    tiles=TILES,
    years=(),
    water_years=WATER_YEARS,
    dates=DATES,
    date_glob=DATE_GLOB,
    dry_run=DRY_RUN,
    apply_valid_inversion_mask=APPLY_VALID_INVERSION_MASK,
    slurm_profile=SlurmProfile(mem=SLURM_MEM, extra_args=(SLURM_EXCLUDE_ARG,)),
).canonicalized()

config


{
  "modis": [
    "terra",
    "aqua"
  ],
  "viirs": [
    "snpp",
    "noaa20",
    "noaa21"
  ]
}



CurcWorkflowConfig(scratch_root=PosixPath('/scratch/alpine/ropa5718/spipy'), input_source_root=PosixPath('/pl/active/rittger_ops/vnp09ga.002'), sensor='viirs', platforms=('snpp',), tiles=('h08v05',), years=(), water_years=(2023,), dates=(), date_glob='*', dry_run=True, max_auto_retry_count=3, apply_valid_inversion_mask=False, use_grouping=True, grouping_method='chunk_bin_mean', slurm_profile=SlurmProfile(account=None, qos=None, time=None, mem='8G', cpus_per_task=None, output_dir=None, extra_args=('--exclude=bgpu-biokem1,bgpu-biokem2,bgpu-biokem3,bgpu-bortz1,bgpu-curc2,bgpu-curc4,bgpu-g4-18,bgpu-g4-u20,bgpu-g4-u24,bgpu-g6-u20,bgpu-ivc,bhpc-c5-u31-1,blanca-g4-u14-3,bmem-rico1,bnode0108,bgpu-g6-u34,bhpc-c5-u7-12,bnode0414,bnode0508,bhpc-c7-u7-8,bgpu-g6-u25,bgpu-chbe-rdi1',)))

## Scratch Layout

The CURC workflow keeps shared staged inputs under `/scratch/alpine/.../input` and dated inversion outputs under `/scratch/alpine/.../output`.

In [3]:
platform = config.platforms[0]
tile = config.tiles[0]
water_year = config.water_years[0]

print("reflectance:", reflectance_dir(config, platform) / tile / str(water_year))
print("ancillary:", ancillary_dir(config, platform) / tile)
print("r0:", r0_dir(config, platform) / tile / str(water_year))
print("output_raw_wy:", output_raw_water_year_root(config, platform, tile, water_year))
print("logs:", log_root(config))


reflectance: /scratch/alpine/ropa5718/spipy/input/viirs/snpp/reflectance/h08v05/2023
ancillary: /scratch/alpine/ropa5718/spipy/input/viirs/snpp/ancillary/h08v05
r0: /scratch/alpine/ropa5718/spipy/input/viirs/snpp/ancillary/r0/h08v05/2023
output_raw_wy: /scratch/alpine/ropa5718/spipy/output/viirs/snpp/h08v05/raw/wy2023
logs: /scratch/alpine/ropa5718/spipy/logs


## Full Water-Year Discovery

This discovery step is the first concrete replacement for the opaque batch bash scripts. It gives a direct view of which source files are available for one tile and one water year.

In [4]:
reflectance_files = discover_viirs_snpp_water_year_reflectance_files(
    config,
    tile=tile,
    water_year=water_year,
)

print("water_year_file_count", len(reflectance_files))
print("first_file", reflectance_files[0] if reflectance_files else None)
print("last_file", reflectance_files[-1] if reflectance_files else None)


water_year_file_count 364
first_file /pl/active/rittger_ops/vnp09ga.002/input/h08v05/2022/VNP09GA.A2022274.h08v05.002.2023067084622.h5
last_file /pl/active/rittger_ops/vnp09ga.002/input/h08v05/2023/VNP09GA.A2023273.h08v05.002.2023276002621.h5


## Full Water-Year Step Plan

The workflow should be step-oriented rather than all-or-nothing. For a full water year, the planner expands the request into the major steps that will later become executable or submitted jobs.

In [5]:
full_water_year_steps = plan_viirs_snpp_steps(
    config,
    tile=tile,
    water_year=water_year,
)

for step in full_water_year_steps:
    summary = {
        "step": step.step,
        "date_count": step.date_count,
        "destination_path": step.destination_path,
        "r0_year": step.r0_year,
    }
    print(json.dumps(summary, indent=2))


{
  "step": "stage_reflectance",
  "date_count": 364,
  "destination_path": "/scratch/alpine/ropa5718/spipy/input/viirs/snpp/reflectance/h08v05/2023",
  "r0_year": null
}
{
  "step": "stage_ancillary",
  "date_count": 364,
  "destination_path": "/scratch/alpine/ropa5718/spipy/input/viirs/snpp/ancillary/h08v05",
  "r0_year": null
}
{
  "step": "build_r0",
  "date_count": 106,
  "destination_path": "/scratch/alpine/ropa5718/spipy/input/viirs/snpp/ancillary/r0/h08v05/2022",
  "r0_year": 2022
}
{
  "step": "run_inversion",
  "date_count": 364,
  "destination_path": "/scratch/alpine/ropa5718/spipy/output/viirs/snpp/h08v05/raw/wy2023",
  "r0_year": 2022
}


## Single-Date Rerun Plan

A key design goal is that a user can rerun a single date without relaunching the entire water year. This is the concrete planning path for that case.

In [6]:
target_dates = ()

target_date_steps = plan_viirs_snpp_steps(
    config,
    tile=tile,
    water_year=water_year,
    target_dates=target_dates,
)

for step in target_date_steps:
    print(json.dumps(asdict(step), indent=2))


{
  "step": "stage_reflectance",
  "sensor": "viirs",
  "platform": "snpp",
  "tile": "h08v05",
  "water_year": 2023,
  "date_count": 5,
  "dates": [
    "2023-03-16",
    "2023-03-17",
    "2023-03-18",
    "2023-03-19",
    "2023-03-20"
  ],
  "source_paths": [
    "/pl/active/rittger_ops/vnp09ga.002/input/h08v05/2023/VNP09GA.A2023075.h08v05.002.2023101231908.h5",
    "/pl/active/rittger_ops/vnp09ga.002/input/h08v05/2023/VNP09GA.A2023076.h08v05.002.2023102004128.h5",
    "/pl/active/rittger_ops/vnp09ga.002/input/h08v05/2023/VNP09GA.A2023077.h08v05.002.2023102020028.h5",
    "/pl/active/rittger_ops/vnp09ga.002/input/h08v05/2023/VNP09GA.A2023078.h08v05.002.2023102033212.h5",
    "/pl/active/rittger_ops/vnp09ga.002/input/h08v05/2023/VNP09GA.A2023079.h08v05.002.2023102063055.h5"
  ],
  "destination_path": "/scratch/alpine/ropa5718/spipy/input/viirs/snpp/reflectance/h08v05/2023",
  "notes": [
    "Copy or rsync the discovered VNP09GA files from /pl to /scratch/alpine.",
    "This step can

## Development Direction

The next implementation steps are not notebook-specific. They belong in `workflows/curc` and should later be callable from notebooks, scripts, and Slurm entrypoints:
- execute `stage_reflectance` as an explicit rsync/copy operation,
- execute `stage_ancillary` with clear checks for static tile inputs,
- implement summer-only source selection for `build_r0`,
- run inversion per date or per chunk while preserving the same step boundaries.

## Step Execution Helpers

Preview direct staging and R0 execution from the current CURC workflow helpers. These steps run outside Slurm; `run_inversion` still uses the manifest-backed array submission flow.


In [7]:
from workflows.curc.runner import preview_viirs_snpp_step_execution, run_viirs_snpp_step

for step_name in ("stage_reflectance", "stage_ancillary"):
    payload = preview_viirs_snpp_step_execution(
        config,
        tile=tile,
        water_year=water_year,
        step=step_name,
        target_dates=target_dates,
    )
    print(f"\n[{step_name}]")
    print(json.dumps(payload, indent=2))



[stage_reflectance]
{
  "step": "stage_reflectance",
  "sensor": "viirs",
  "platform": "snpp",
  "tile": "h08v05",
  "water_year": 2023,
  "date_count": 5,
  "dates": [
    "2023-03-16",
    "2023-03-17",
    "2023-03-18",
    "2023-03-19",
    "2023-03-20"
  ],
  "source_path_count": 5,
  "source_paths": [
    "/pl/active/rittger_ops/vnp09ga.002/input/h08v05/2023/VNP09GA.A2023075.h08v05.002.2023101231908.h5",
    "/pl/active/rittger_ops/vnp09ga.002/input/h08v05/2023/VNP09GA.A2023076.h08v05.002.2023102004128.h5",
    "/pl/active/rittger_ops/vnp09ga.002/input/h08v05/2023/VNP09GA.A2023077.h08v05.002.2023102020028.h5",
    "/pl/active/rittger_ops/vnp09ga.002/input/h08v05/2023/VNP09GA.A2023078.h08v05.002.2023102033212.h5",
    "/pl/active/rittger_ops/vnp09ga.002/input/h08v05/2023/VNP09GA.A2023079.h08v05.002.2023102063055.h5"
  ],
  "destination_path": "/gpfs/alpine1/scratch/ropa5718/spipy/input/viirs/snpp/reflectance/h08v05/2023",
  "notes": [
    "Copy or rsync the discovered VNP09GA fi

In [8]:
# Set EXECUTE_STAGE_REFLECTANCE = True when you want to rsync the targeted reflectance files to scratch.
EXECUTE_STAGE_REFLECTANCE = True

stage_reflectance_result = run_viirs_snpp_step(
    config,
    tile=tile,
    water_year=water_year,
    step="stage_reflectance",
    target_dates=target_dates,
    execute=EXECUTE_STAGE_REFLECTANCE,
    overwrite=False,
    show_progress=True,
)
print(json.dumps(stage_reflectance_result, indent=2))


{
  "step": "stage_reflectance",
  "sensor": "viirs",
  "platform": "snpp",
  "tile": "h08v05",
  "water_year": 2023,
  "date_count": 5,
  "dates": [
    "2023-03-16",
    "2023-03-17",
    "2023-03-18",
    "2023-03-19",
    "2023-03-20"
  ],
  "source_path_count": 5,
  "source_paths": [
    "/pl/active/rittger_ops/vnp09ga.002/input/h08v05/2023/VNP09GA.A2023075.h08v05.002.2023101231908.h5",
    "/pl/active/rittger_ops/vnp09ga.002/input/h08v05/2023/VNP09GA.A2023076.h08v05.002.2023102004128.h5",
    "/pl/active/rittger_ops/vnp09ga.002/input/h08v05/2023/VNP09GA.A2023077.h08v05.002.2023102020028.h5",
    "/pl/active/rittger_ops/vnp09ga.002/input/h08v05/2023/VNP09GA.A2023078.h08v05.002.2023102033212.h5",
    "/pl/active/rittger_ops/vnp09ga.002/input/h08v05/2023/VNP09GA.A2023079.h08v05.002.2023102063055.h5"
  ],
  "destination_path": "/gpfs/alpine1/scratch/ropa5718/spipy/input/viirs/snpp/reflectance/h08v05/2023",
  "notes": [
    "Copy or rsync the discovered VNP09GA files from /pl to /scra

In [ ]:
import xarray as xr

test = xr.open_dataset('/scratch/alpine/ropa5718/spipy/output/viirs/snpp/h08v05/raw/wy2023/snpp_raw_output_h08v05_20230316.nc')
test['raw_viewable_snow_fraction'].plot(figsize=(24,20), cmap='YlGnBu_r')


In [ ]:
test

## Slurm Inversion Submission (Full Water Year)

Use these cells to create a full-water-year manifest for tile `h08v05`, submit the inversion array, and inspect scanner output/log files.

Run the `stage_reflectance` execution cell above first so the water-year reflectance set exists under scratch before submitting the inversion array.


In [9]:
from pathlib import Path
import json
import subprocess
import sys

from workflows.curc.runner import plan_viirs_snpp_inversion_array_submission

repo_root = Path.cwd()
if not (repo_root / "scripts" / "submit_curc_inversion_array.py").exists():
    repo_root = repo_root.parent
target_dates = ()
target_r0_year = 2022
target_max_concurrent_tasks = None

array_payload = plan_viirs_snpp_inversion_array_submission(
    config,
    tile=tile,
    water_year=water_year,
    target_dates=target_dates,
    r0_year=target_r0_year,
    max_concurrent_tasks=target_max_concurrent_tasks,
)

print(json.dumps(array_payload, indent=2))
print(f"\nmanifest_path={array_payload['manifest_path']}")


{
  "step": "run_inversion",
  "job_name": "spipy-viirs-snpp-h08v05-wy2023",
  "sensor": "viirs",
  "platform": "snpp",
  "tile": "h08v05",
  "water_year": 2023,
  "task_count": 5,
  "array_indices": [
    0,
    1,
    2,
    3,
    4
  ],
  "max_concurrent_tasks": 1,
  "max_auto_retry_count": 3,
  "apply_valid_inversion_mask": false,
  "use_grouping": true,
  "grouping_method": "chunk_bin_mean",
  "notes": [
    "Logical inversion unit is one acquisition date.",
    "Submission unit is one Slurm array spanning those dates."
  ],
  "r0_year": 2022,
  "slurm_profile": {
    "account": null,
    "qos": null,
    "time": null,
    "mem": "8G",
    "cpus_per_task": null,
    "output_dir": null,
    "extra_args": [
      "--exclude=bgpu-biokem1,bgpu-biokem2,bgpu-biokem3,bgpu-bortz1,bgpu-curc2,bgpu-curc4,bgpu-g4-18,bgpu-g4-u20,bgpu-g4-u24,bgpu-g6-u20,bgpu-ivc,bhpc-c5-u31-1,blanca-g4-u14-3,bmem-rico1,bnode0108,bgpu-g6-u34,bhpc-c5-u7-12,bnode0414,bnode0508,bhpc-c7-u7-8,bgpu-g6-u25,bgpu-chbe-r

In [10]:
submit_cmd = [
    sys.executable,
    str(repo_root / "scripts" / "submit_curc_inversion_array.py"),
    array_payload["manifest_path"],
    "--submit",
    "--python-exec",
    sys.executable,
    "--execution-profile",
    "cluster",
]

print("Running:", " ".join(submit_cmd))
submit_completed = subprocess.run(submit_cmd, check=True, text=True, capture_output=True)
submit_result = json.loads(submit_completed.stdout)
print(json.dumps(submit_result, indent=2))

if submit_result.get("submitted"):
    print("\nSubmitted array job:", submit_result.get("sbatch_stdout", "").strip())


Running: /projects/ropa5718/software/anaconda/envs/spipy14/bin/python /projects/ropa5718/SpiPy/scripts/submit_curc_inversion_array.py /gpfs/alpine1/scratch/ropa5718/spipy/logs/20260519_134558/spipy-viirs-snpp-h08v05-wy2023_array_manifest.json --submit --python-exec /projects/ropa5718/software/anaconda/envs/spipy14/bin/python --execution-profile cluster
{
  "manifest_path": "/gpfs/alpine1/scratch/ropa5718/spipy/logs/20260519_134558/spipy-viirs-snpp-h08v05-wy2023_array_manifest.json",
  "report": {
    "manifest_path": "/gpfs/alpine1/scratch/ropa5718/spipy/logs/20260519_134558/spipy-viirs-snpp-h08v05-wy2023_array_manifest.json",
    "task_count": 5,
    "completed_count": 1,
    "failed_count": 4,
    "retryable_count": 4,
    "auto_retry_eligible_count": 4,
    "retry_exhausted_count": 0,
    "missing_count": 4,
    "max_auto_retry_count": 3,
    "tasks": [
      {
        "task_index": 0,
        "date": "2023-03-16",
        "retry_count": 0,
        "status": "completed",
        "fa

In [19]:
submitted_job_id = (submit_result.get("sbatch_stdout", "") or "").strip().split(";")[0]
if submitted_job_id:
    sacct_cmd = [
        "sacct",
        "-j",
        submitted_job_id,
        "--format=JobID,JobName%35,State,ExitCode,Elapsed,MaxRSS,ReqMem,NodeList",
        "-P",
    ]
    print("Running:", " ".join(sacct_cmd))
    sacct_completed = subprocess.run(sacct_cmd, check=False, text=True, capture_output=True)
    print(sacct_completed.stdout)
    if sacct_completed.stderr.strip():
        print("stderr:\n", sacct_completed.stderr)
else:
    print("No submitted job id found in submit_result['sbatch_stdout']")


Running: sacct -j 26186351 --format=JobID,JobName%35,State,ExitCode,Elapsed,MaxRSS,ReqMem,NodeList -P
JobID|JobName|State|ExitCode|Elapsed|MaxRSS|ReqMem|NodeList
26186351_0|spipy-viirs-snpp-h08v05-wy2023|COMPLETED|0:0|00:00:05||8G|bnode0404
26186351_0.batch|batch|COMPLETED|0:0|00:00:05|236620K||bnode0404
26186351_0.extern|extern|COMPLETED|0:0|00:00:05|284K||bnode0404
26186351_1|spipy-viirs-snpp-h08v05-wy2023|COMPLETED|0:0|00:03:39||8G|bnode0404
26186351_1.batch|batch|COMPLETED|0:0|00:03:39|2300184K||bnode0404
26186351_1.extern|extern|COMPLETED|0:0|00:03:39|284K||bnode0404
26186351_2|spipy-viirs-snpp-h08v05-wy2023|COMPLETED|0:0|00:03:50||8G|bnode0404
26186351_2.batch|batch|COMPLETED|0:0|00:03:50|2321320K||bnode0404
26186351_2.extern|extern|COMPLETED|0:0|00:03:50|284K||bnode0404
26186351_3|spipy-viirs-snpp-h08v05-wy2023|COMPLETED|0:0|00:04:56||8G|bnode0404
26186351_3.batch|batch|COMPLETED|0:0|00:04:56|2333188K||bnode0404
26186351_3.extern|extern|COMPLETED|0:0|00:04:56|284K||bnode0404
261

In [ ]:
scan_cmd = [
    sys.executable,
    str(repo_root / "scripts" / "scan_curc_inversion_array.py"),
    array_payload["manifest_path"],
]

print("Running:", " ".join(scan_cmd))
scan_completed = subprocess.run(scan_cmd, check=True, text=True, capture_output=True)
scan_result = json.loads(scan_completed.stdout)
print(json.dumps(scan_result, indent=2))


In [ ]:
log_root = config.scratch_root / "logs"
manifest_dir = Path(array_payload["manifest_path"]).parent

print("Top-level log root:", log_root)
for path in sorted(log_root.glob("run_inversion*")):
    print(path)

print("\nManifest-side files:", manifest_dir)
for path in sorted(manifest_dir.glob("*inversion*")):
    print(path)
